# Phase 2 — Rerank (K1 features + K2 LightGBM LambdaMART)

Trains K2 on the train split (serve-identical query+fusion incl. dense-text), saves the model to Drive, and reports reranked vs fusion-only nDCG@20 on dev. Spec: `51_K2_lgbm_lambdamart.md`, `50_K1_*`.

## 1. Drive + HF auth (Colab Secrets)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/recsys2026'
os.environ['HF_HOME']=f'{DRIVE}/hf_cache'; OUT=f'{DRIVE}/outputs'
os.makedirs(os.environ['HF_HOME'],exist_ok=True); os.makedirs(OUT,exist_ok=True)
_hf=(lambda n:(userdata.get(n) if True else None))
try:
    t=userdata.get('HF_TOKEN'); os.environ['HF_TOKEN']=os.environ['HUGGINGFACE_HUB_TOKEN']=t
    from huggingface_hub import login; login(t); print('HF ok')
except Exception as e: print('no HF_TOKEN secret:', e)

## 2. Clone + install

In [ ]:
!git clone --branch fresh-start --depth 1 https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026 2>/dev/null || (cd /content/recsys2026 && git pull)
%cd /content/recsys2026
!pip -q install datasets bm25s scipy scikit-learn lightgbm sentence-transformers numpy pandas
import sys; sys.path.insert(0,'.')

## 3. Config

In [ ]:
TOPK=500; TRAIN_SESSIONS=3000; DEV_SESSIONS=1000
NEG_CAP=150            # random-sampled negatives/group (0 = all); preserves score dist
EARLY_STOPPING=50      # session-disjoint val + early stop on val ndcg@20 (0 = off)
DENSE_MODEL='BAAI/bge-large-en-v1.5'; CONTENT='metadata-qwen3_embedding_0.6b'; ORG='talkpl-ai'

## 4. Load data (HF) + build channels incl. dense

In [ ]:
from datasets import load_dataset
from mcrs.data.catalog import Catalog
from mcrs.data.embeddings import TrackEmbeddings, UserEmbeddings
from mcrs.data.conversations import Conversations
from sentence_transformers import SentenceTransformer
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.retrieval.dense_channel import DenseChannel
from mcrs.retrieval.personalization import ContentKNNChannel, CFChannel, SameArtistChannel
from mcrs.retrieval.fusion import RRFFusion

cat=Catalog(load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Metadata',split='all_tracks'))
tre=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Embeddings',split='all_tracks')
te_content=TrackEmbeddings(tre.select_columns(['track_id',CONTENT]),modalities=[CONTENT])
te_cf=TrackEmbeddings(tre.select_columns(['track_id','cf-bpr']),modalities=['cf-bpr'])
ued=load_dataset(f'{ORG}/TalkPlayData-Challenge-User-Embeddings'); ue=UserEmbeddings([r for sp in ued for r in ued[sp]])
dsd=load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset')
conv_tr=Conversations(dsd['train'].select(range(TRAIN_SESSIONS))); conv_dv=Conversations(dsd['test'].select(range(DEV_SESSIONS)))

model=SentenceTransformer(DENSE_MODEL,device='cuda')
doc_mat=model.encode([cat.id_to_metadata(t) for t in cat.index_to_id],batch_size=256,normalize_embeddings=True,show_progress_bar=True)
dense=DenseChannel(cat.index_to_id,doc_mat,lambda qs:model.encode(qs,batch_size=256,normalize_embeddings=True),normalize=False)
chans=[BM25Channel(cat),dense,ContentKNNChannel(te_content,CONTENT),CFChannel(ue,te_cf,'cf-bpr'),SameArtistChannel(cat)]
fusion=RRFFusion(chans,k=60); labels=[c.label for c in chans]

## 5. Train K2 + save model to Drive

In [ ]:
from mcrs.rerank.features import FeatureBuilder
from mcrs.rerank.lgbm import LGBMReranker
from mcrs.rerank.train import build_rerank_groups
qb=QueryBuilder(); fb=FeatureBuilder(cat,labels)
tr=list(conv_tr.turns())
print('train turns:',len(tr),'- building groups (runs fusion incl dense over train)...')
groups=build_rerank_groups(qb,fusion,tr,lambda t:conv_tr.gold(t.session_id,t.turn_number),topk=TOPK)
k2=LGBMReranker(fb,n_estimators=500,neg_cap=NEG_CAP,early_stopping_rounds=EARLY_STOPPING,val_fraction=0.1).fit(groups)
print(f'train_groups={k2.n_train_groups_} val_groups={k2.n_val_groups_} best_iter={getattr(k2.model,"best_iteration_",None)}')
k2.save(f'{OUT}/k2_lgbm.txt'); print('saved model ->',f'{OUT}/k2_lgbm.txt')
print('feature importances:',sorted(zip(fb.feature_names,k2.model.feature_importances_),key=lambda x:-x[1])[:8])

## 6. Eval reranked vs fusion-only (dev) → save scores

In [ ]:
import json
from mcrs.filter.assembly import TopKAssembler
from mcrs.run.harness import InferenceHarness, validate_submission
from mcrs.eval.harness import GoldRow
from mcrs.eval.official import score_official
dv=list(conv_dv.turns())
golds=[GoldRow(t.session_id,t.user_id,t.turn_number,conv_dv.gold(t.session_id,t.turn_number)) for t in dv]
keys=[(g.session_id,g.turn_number) for g in golds]
base=InferenceHarness(qb,fusion,TopKAssembler(cat),topk=TOPK).run(dv)
rer =InferenceHarness(qb,fusion,TopKAssembler(cat),reranker=k2,topk=TOPK).run(dv)
validate_submission(rer,catalog=cat,expected_keys=keys)
sb=score_official(base,golds,len(cat)); sr=score_official(rer,golds,len(cat))
print('fusion-only nDCG@20=',round(sb['ndcg@20'],4),'| +K2=',round(sr['ndcg@20'],4))
json.dump({'fusion':sb,'reranked':sr}, open(f'{OUT}/phase2_scores.json','w'), indent=2)
print('saved ->',f'{OUT}/phase2_scores.json')

## 7. Next
Gate K2: dev nDCG@20 should clearly beat fusion-only. The absolute ceiling is bounded by fused recall@K (P0: ~0.53) — to push toward 0.55, lift recall (A1 enrichment / R6 / stronger dense), then retrain K2.